In [2]:
!pip install librosa

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.1 MB/s  0:00:00

   ------------- -------------------------- 2/6 [audioread]
   -------------------------- ------------- 4/6 [pooch]
   -------------------------- ------------- 4/6 [pooch]
   --------------------------------- ------ 5/6 [librosa]
   --------------------------------- ------ 5/6 [librosa]
   --------------------------------- ------ 5/6 [librosa]
   --------------------------------- ------ 5/6 [librosa]
   ---------------------------------------- 6/6 [librosa]




[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import librosa

def extract_mfcc(file_path, max_len=20):
    y, sr = librosa.load(file_path, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc = np.pad(mfcc, ((0,0),(0,max_len-mfcc.shape[1])), mode='constant') if mfcc.shape[1] < max_len else mfcc[:, :max_len]
    return mfcc.T

X = np.random.rand(100, 20, 13)
y = np.random.randint(1, 10, (100, 10))

vocab_size, embedding_dim, latent_dim = 15, 16, 32

encoder_inputs = layers.Input(shape=(20, 13))
_, state_h, state_c = layers.LSTM(latent_dim, return_state=True)(encoder_inputs)
encoder_states = [state_h, state_c]

decoder_inputs = layers.Input(shape=(9,))
x = layers.Embedding(vocab_size, embedding_dim)(decoder_inputs)
decoder_outputs, _, _ = layers.LSTM(latent_dim, return_sequences=True, return_state=True)(x, initial_state=encoder_states)
decoder_outputs = layers.Dense(vocab_size, activation='softmax')(decoder_outputs)

model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

decoder_input_data, decoder_target_data = y[:, :-1], y[:, 1:]
model.fit([X, decoder_input_data], decoder_target_data, epochs=20, batch_size=32)

test_input = X[0:1]
target_seq = np.array([[1]])
decoded_sentence = []

for _ in range(10):
    output_tokens = model.predict([test_input, target_seq], verbose=0)
    sampled_token = np.argmax(output_tokens[0, -1, :])
    decoded_sentence.append(sampled_token)
    target_seq = np.append(target_seq, [[sampled_token]], axis=1)

print("Generated Tokens:", decoded_sentence)


Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.0878 - loss: 2.7095
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1156 - loss: 2.6725
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1056 - loss: 2.6376
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1078 - loss: 2.5990
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1100 - loss: 2.5549
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1033 - loss: 2.5039 
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1044 - loss: 2.4502
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1044 - loss: 2.4034
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1078 - loss: 2.3627
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1033 - loss: 2.3279
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1200 - loss: 2.2995
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1222 - loss: 2.2773
E

In [7]:
# Install if needed
# !pip install librosa

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model

# -----------------------------
# Dummy Vocabulary
# -----------------------------
word_index = {
    1: "<start>",
    2: "hello",
    3: "world",
    4: "yes",
    5: "no",
    6: "data",
    7: "science",
    8: "ai",
    9: "model"
}

index_word = {k: v for k, v in word_index.items()}

vocab_size = 15
embedding_dim = 16
latent_dim = 64

# -----------------------------
# Dummy Dataset (Replace later)
# -----------------------------
X = np.random.rand(100, 20, 13)
y = np.random.randint(1, 10, (100, 10))

# -----------------------------
# Encoder
# -----------------------------
encoder_inputs = layers.Input(shape=(20, 13))
encoder_lstm = layers.LSTM(latent_dim, return_state=True)

_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# -----------------------------
# Decoder
# -----------------------------
decoder_inputs = layers.Input(shape=(None,))
decoder_embedding = layers.Embedding(vocab_size, embedding_dim)

decoder_lstm = layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_dense = layers.Dense(vocab_size, activation='softmax')

x = decoder_embedding(decoder_inputs)
decoder_outputs, _, _ = decoder_lstm(x, initial_state=encoder_states)
decoder_outputs = decoder_dense(decoder_outputs)

# -----------------------------
# Model
# -----------------------------
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# Prepare Data (Teacher Forcing)
# -----------------------------
decoder_input_data = y[:, :-1]
decoder_target_data = y[:, 1:]

# -----------------------------
# Train
# -----------------------------
model.fit(
    [X, decoder_input_data],
    decoder_target_data,
    epochs=20,
    batch_size=32
)

# -----------------------------
# Inference (Generate Text)
# -----------------------------
test_input = X[0:1]

target_seq = np.array([[1]])  # start token
decoded_sentence = []

for _ in range(10):
    output_tokens = model.predict([test_input, target_seq], verbose=0)

    sampled_token = int(np.argmax(output_tokens[0, -1, :]))
    decoded_sentence.append(sampled_token)

    target_seq = np.append(target_seq, [[sampled_token]], axis=1)

# -----------------------------
# Convert Tokens to Words
# -----------------------------
decoded_words = [index_word.get(token, "?") for token in decoded_sentence]

print("Generated Tokens:", decoded_sentence)
print("Generated Text:", " ".join(decoded_words))

Epoch 1/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.1211 - loss: 2.6770
Epoch 2/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1178 - loss: 2.5997
Epoch 3/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1233 - loss: 2.4930
Epoch 4/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1222 - loss: 2.3773
Epoch 5/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1233 - loss: 2.3036
Epoch 6/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1156 - loss: 2.2631
Epoch 7/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1178 - loss: 2.2438
Epoch 8/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1078 - loss: 2.2333
Epoch 9/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1100 - loss: 2.2205
Epoch 10/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1322 - loss: 2.2093
Epoch 11/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.1356 - loss: 2.2047
Epoch 12/20
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1300 - loss: 2.2023
E